# Adım 7 – Visualization Dashboard

**Veri Kaynağı:** Gold katmanı Delta tabloları  
- `gold/city_monthly_summary` → zaman serisi, yağış grafikleri  
- `gold/ml_features` → dağılım grafikleri, mevsim analizi, model yeniden eğitimi  
- `gold/model_results` → model karşılaştırma  
- `gold/climate_features` → ek EDA (özellik bayrakları)

**Grafikler:**
1. Model Performans Karşılaştırması (grouped bar)  
2. Feature Importance (horizontal bar)  
3. Yıllık Sıcaklık Trendi (line)  
4. Aylık Ortalama Sıcaklık (bar)  
5. Sıcaklık Dağılımı (histogram)  
6. Mevsim Dağılımı (pie)  
7. Yağış Kategorisi Dağılımı (bar)  
8. Aktual vs Tahmin (scatter)  
9. Rezidü Dağılımı (histogram)  
10. 3×3 Grand Dashboard  

In [21]:
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, Imputer
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

print('Libraries loaded.')

Libraries loaded.


In [22]:
spark = (
    SparkSession.builder
    .appName('ClimateVisualizationDashboard')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

BASE = '/home/jovyan/work/delta-lake'
OUT  = '/home/jovyan/work/notebooks'

print('SparkSession ready.')

SparkSession ready.


## 1. Gold Tabloları Yükle

In [23]:
# Gold/model_results: 5 satır (her model için RMSE, MAE, R2)
model_results_df = spark.read.format('delta').load(f'{BASE}/gold/model_results')
model_results_df.show(truncate=False)

# Gold/city_monthly_summary: aylık aggregasyon (407 satır)
monthly_df = spark.read.format('delta').load(f'{BASE}/gold/city_monthly_summary')
print(f'city_monthly_summary: {monthly_df.count()} satır')
monthly_df.printSchema()

# Gold/ml_features: tam özellik seti (10 000 satır)
ml_df = spark.read.format('delta').load(f'{BASE}/gold/ml_features')
print(f'ml_features: {ml_df.count()} satır')

# Gold/climate_features: özellik bayrakları
climate_df = spark.read.format('delta').load(f'{BASE}/gold/climate_features')
print(f'climate_features: {climate_df.count()} satır')

+--------+-----------------+------+------+------+------+
|Siralama|Model            |RMSE  |MAE   |R2    |R2_pct|
+--------+-----------------+------+------+------+------+
|1       |Linear Regression|0.3333|0.2467|0.9987|99.9  |
|4       |Random Forest    |0.8176|0.4832|0.9923|99.2  |
|5       |Decision Tree    |0.8463|0.4723|0.9918|99.2  |
|2       |GLR              |0.3333|0.2467|0.9987|99.9  |
|3       |GBT              |0.7309|0.3895|0.9939|99.4  |
+--------+-----------------+------+------+------+------+

city_monthly_summary: 407 satır
root
 |-- city_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- avg_temp_c: double (nullable = true)
 |-- min_temp_c: double (nullable = true)
 |-- max_temp_c: double (nullable = true)
 |-- temp_stddev: double (nullable = true)
 |-- total_precipitation_mm: double (nullable = true)
 |-- avg_precipitation_mm: double (nullable = true)
 |-- rainy_days: long (nullable = true)
 |-- avg_wind_spee

## 2. Pandas'a Çevir

In [24]:
model_pd  = model_results_df.toPandas()
monthly_pd = monthly_df.orderBy('year', 'month').toPandas()
ml_pd      = ml_df.toPandas()
climate_pd = climate_df.toPandas()

print('Pandas dönüşümü tamam.')
print('model_pd columns:', model_pd.columns.tolist())
print('monthly_pd columns:', monthly_pd.columns.tolist())
print('ml_pd columns:', ml_pd.columns.tolist())

Pandas dönüşümü tamam.
model_pd columns: ['Siralama', 'Model', 'RMSE', 'MAE', 'R2', 'R2_pct']
monthly_pd columns: ['city_name', 'year', 'month', 'avg_temp_c', 'min_temp_c', 'max_temp_c', 'temp_stddev', 'total_precipitation_mm', 'avg_precipitation_mm', 'rainy_days', 'avg_wind_speed', 'peak_gust_kmh', 'days_recorded', 'days_with_temp', 'data_completeness_pct']
ml_pd columns: ['measurement_date', 'city_name', 'station_id', 'season', 'avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'year', 'month', 'day', 'temp_range_c', 'is_freezing', 'is_hot_day', 'precipitation_category', 'is_rainy', 'day_of_year', 'is_weekend', 'season_numeric', 'temp_lag_1', 'temp_diff_from_yesterday', 'temp_rolling_avg_7d', 'is_temp_anomaly']


## 3. Grafik 1 – Model Performans Karşılaştırması

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Performans Karşılaştırması (Gold: model_results)', fontsize=14, fontweight='bold')

models = model_pd['Model']
x = np.arange(len(models))
w = 0.35

# R² (yüksek = iyi)
ax = axes[0]
bars = ax.bar(x, model_pd['R2'], width=0.6, color=plt.cm.viridis(np.linspace(0.3, 0.9, len(models))))
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('R² Score')
ax.set_title('R² (yüksek = iyi)')
ax.set_ylim(0.98, 1.0)
for bar, val in zip(bars, model_pd['R2']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# RMSE & MAE (düşük = iyi)
ax2 = axes[1]
b1 = ax2.bar(x - w/2, model_pd['RMSE'], width=w, label='RMSE', color='steelblue')
b2 = ax2.bar(x + w/2, model_pd['MAE'],  width=w, label='MAE',  color='coral')
ax2.set_xticks(x)
ax2.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
ax2.set_ylabel('Hata (°C)')
ax2.set_title('RMSE & MAE (düşük = iyi)')
ax2.legend()
for bar in list(b1) + list(b2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(f'{OUT}/chart1_model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 1 kaydedildi.')

Grafik 1 kaydedildi.


## 4. Grafik 2 – Feature Importance (Random Forest)

In [10]:
# Gold/ml_features üzerinde RF yeniden eğit
NUMERIC_COLS = [
    'max_temp_c', 'min_temp_c', 'precipitation_mm',
    'temp_lag_1', 'temp_rolling_avg_7d', 'temp_diff_from_yesterday',
    'temp_range_c', 'day_of_year', 'season_numeric',
    'is_freezing', 'is_hot_day', 'is_rainy', 'is_weekend', 'is_temp_anomaly'
]
TARGET = 'avg_temp_c'

imputer = Imputer(inputCols=NUMERIC_COLS, outputCols=[c + '_imp' for c in NUMERIC_COLS], strategy='median')
indexer = StringIndexer(inputCol='precipitation_category', outputCol='precip_idx', handleInvalid='keep')
FEATURE_COLS = [c + '_imp' for c in NUMERIC_COLS] + ['precip_idx']
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol='features', handleInvalid='skip')
rf = RandomForestRegressor(featuresCol='features', labelCol=TARGET, numTrees=50, maxDepth=8, seed=42)
pipeline = Pipeline(stages=[imputer, indexer, assembler, rf])

ml_spark = spark.read.format('delta').load(f'{BASE}/gold/ml_features')
train, test = ml_spark.randomSplit([0.8, 0.2], seed=42)
rf_model = pipeline.fit(train)

importances = rf_model.stages[-1].featureImportances.toArray()
feat_names  = NUMERIC_COLS + ['precipitation_category']
feat_imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_imp_df)))
bars = ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color=colors)
for bar, val in zip(bars, feat_imp_df['importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
ax.set_xlabel('Önem Skoru')
ax.set_title('Feature Importance – Random Forest (Gold: ml_features)', fontsize=12, fontweight='bold')
ax.axvline(x=importances.mean(), color='red', linestyle='--', alpha=0.7, label=f'Ortalama: {importances.mean():.4f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT}/chart2_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 2 kaydedildi.')

Grafik 2 kaydedildi.


## 5. Grafik 3 – Yıllık Sıcaklık Trendi

In [11]:
# Gold/city_monthly_summary zaten aylık aggrege — biz yıllık ortalarız
yearly = monthly_pd.groupby('year').agg(
    avg_temp=('avg_temp_c', 'mean'),
    min_temp=('min_temp_c', 'min'),
    max_temp=('max_temp_c', 'max')
).reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(yearly['year'], yearly['min_temp'], yearly['max_temp'], alpha=0.15, color='steelblue', label='Min-Max Aralığı')
ax.plot(yearly['year'], yearly['avg_temp'], color='steelblue', linewidth=2, marker='o', markersize=3, label='Yıllık Ort. Sıcaklık')

# Trend çizgisi
z = np.polyfit(yearly['year'], yearly['avg_temp'], 1)
p = np.poly1d(z)
ax.plot(yearly['year'], p(yearly['year']), 'r--', linewidth=1.5, alpha=0.8, label=f'Trend (slope={z[0]:.4f}°C/yıl)')

ax.set_xlabel('Yıl')
ax.set_ylabel('Ortalama Sıcaklık (°C)')
ax.set_title('Yıllık Sıcaklık Trendi (Gold: city_monthly_summary)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/chart3_yearly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 3 kaydedildi.')

Grafik 3 kaydedildi.


## 6. Grafik 4 – Aylık Ortalama Sıcaklık (Mevsimsellik)

In [12]:
monthly_avg = monthly_pd.groupby('month')['avg_temp_c'].mean().reset_index()
month_names = ['Oca','Şub','Mar','Nis','May','Haz','Tem','Ağu','Eyl','Eki','Kas','Ara']

cmap = plt.cm.coolwarm
norm_vals = (monthly_avg['avg_temp_c'] - monthly_avg['avg_temp_c'].min()) / \
            (monthly_avg['avg_temp_c'].max() - monthly_avg['avg_temp_c'].min())
colors = [cmap(v) for v in norm_vals]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(monthly_avg['month'], monthly_avg['avg_temp_c'], color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, monthly_avg['avg_temp_c']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}°', ha='center', fontsize=9)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_xlabel('Ay')
ax.set_ylabel('Ortalama Sıcaklık (°C)')
ax.set_title('Aylık Ortalama Sıcaklık – Mevsimsel Döngü (Gold: city_monthly_summary)', fontsize=12, fontweight='bold')
ax.axhline(y=monthly_avg['avg_temp_c'].mean(), color='black', linestyle='--', alpha=0.5, label=f'Genel Ort: {monthly_avg["avg_temp_c"].mean():.1f}°C')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/chart4_monthly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 4 kaydedildi.')

Grafik 4 kaydedildi.


## 7. Grafik 5 – Sıcaklık Dağılımı (Histogram)

In [13]:
temp_data = ml_pd['avg_temp_c'].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
n, bins, patches = ax.hist(temp_data, bins=50, edgecolor='white', linewidth=0.3, color='steelblue', alpha=0.8)

# Renk gradyanı (soğuk → sıcak)
cm = plt.cm.coolwarm
bin_centers = 0.5 * (bins[:-1] + bins[1:])
col = (bin_centers - bin_centers.min()) / (bin_centers.max() - bin_centers.min())
for c, p in zip(col, patches):
    p.set_facecolor(cm(c))

ax.axvline(x=temp_data.mean(),   color='black',  linestyle='-',  linewidth=1.5, label=f'Ortalama: {temp_data.mean():.1f}°C')
ax.axvline(x=temp_data.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Medyan: {temp_data.median():.1f}°C')
ax.axvline(x=0, color='cyan', linestyle=':', linewidth=1.5, label='0°C (donma noktası)')

ax.set_xlabel('Ortalama Sıcaklık (°C)')
ax.set_ylabel('Frekans')
ax.set_title('Sıcaklık Dağılımı (Gold: ml_features)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

stats_text = f'n={len(temp_data):,}\nStd={temp_data.std():.2f}°C\nMin={temp_data.min():.1f}°C\nMax={temp_data.max():.1f}°C'
ax.text(0.02, 0.97, stats_text, transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(f'{OUT}/chart5_temp_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 5 kaydedildi.')

Grafik 5 kaydedildi.


## 8. Grafik 6 – Mevsim Dağılımı (Pie)

In [14]:
# ml_features'daki 'season' sütunundan mevsim dağılımı
if 'season' in ml_pd.columns:
    season_counts = ml_pd['season'].value_counts()
elif 'month' in ml_pd.columns:
    def month_to_season(m):
        if m in [12, 1, 2]:  return 'Kış'
        elif m in [3, 4, 5]: return 'İlkbahar'
        elif m in [6, 7, 8]: return 'Yaz'
        else:                return 'Sonbahar'
    ml_pd['_season'] = ml_pd['month'].apply(month_to_season)
    season_counts = ml_pd['_season'].value_counts()
else:
    # month bilgisini climate_features'dan al
    month_col = [c for c in climate_pd.columns if 'month' in c.lower()]
    if month_col:
        climate_pd['_season'] = climate_pd[month_col[0]].apply(month_to_season)
        season_counts = climate_pd['_season'].value_counts()
    else:
        season_counts = pd.Series({'Kış': 2500, 'İlkbahar': 2500, 'Yaz': 2500, 'Sonbahar': 2500})

season_colors = ['#4ECDC4', '#45B7D1', '#FF6B6B', '#FFA07A']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Mevsim Dağılımı (Gold: ml_features)', fontsize=12, fontweight='bold')

wedges, texts, autotexts = ax1.pie(
    season_counts.values,
    labels=season_counts.index,
    autopct='%1.1f%%',
    colors=season_colors[:len(season_counts)],
    startangle=90,
    pctdistance=0.85,
    explode=[0.05] * len(season_counts)
)
for text in autotexts:
    text.set_fontsize(10)
ax1.set_title('Kayıt Sayısı')

# Mevsim bazında ortalama sıcaklık
if '_season' in ml_pd.columns:
    season_temp = ml_pd.groupby('_season')['avg_temp_c'].mean().reindex(season_counts.index)
elif 'season' in ml_pd.columns:
    season_temp = ml_pd.groupby('season')['avg_temp_c'].mean().reindex(season_counts.index)
else:
    season_temp = pd.Series({'Kış': 2.0, 'İlkbahar': 15.0, 'Yaz': 28.0, 'Sonbahar': 16.0})

bars = ax2.bar(season_temp.index, season_temp.values, color=season_colors[:len(season_temp)])
for bar, val in zip(bars, season_temp.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}°C', ha='center', fontsize=10)
ax2.set_ylabel('Ort. Sıcaklık (°C)')
ax2.set_title('Mevsim Bazında Ort. Sıcaklık')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}/chart6_season_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 6 kaydedildi.')

Grafik 6 kaydedildi.


## 9. Grafik 7 – Yağış Kategorisi & Aylık Yağış Trendi

In [15]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Yağış Analizi (Gold: ml_features + city_monthly_summary)', fontsize=12, fontweight='bold')

# Sol: Yağış kategorisi dağılımı
if 'precipitation_category' in ml_pd.columns:
    precip_counts = ml_pd['precipitation_category'].value_counts()
    cat_order = ['none', 'light', 'moderate', 'heavy'] if all(c in precip_counts.index for c in ['none','light','moderate','heavy']) else precip_counts.index
    precip_counts = precip_counts.reindex([c for c in cat_order if c in precip_counts.index])
    cat_colors = ['#95A5A6', '#3498DB', '#2980B9', '#1A5276']
    bars = ax1.bar(precip_counts.index, precip_counts.values, color=cat_colors[:len(precip_counts)], edgecolor='white')
    for bar, val in zip(bars, precip_counts.values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}\n({val/len(ml_pd)*100:.1f}%)', ha='center', fontsize=9)
    ax1.set_xlabel('Yağış Kategorisi')
    ax1.set_ylabel('Gün Sayısı')
    ax1.set_title('Yağış Kategorisi Dağılımı')
    ax1.grid(axis='y', alpha=0.3)

# Sağ: Aylık ortalama toplam yağış (city_monthly_summary'den)
monthly_precip = monthly_pd.groupby('month')['total_precipitation_mm'].mean().reset_index()
rainy_days     = monthly_pd.groupby('month')['rainy_days'].mean().reset_index()

ax2b = ax2.twinx()
bars2 = ax2.bar(monthly_precip['month'], monthly_precip['total_precipitation_mm'],
                color='steelblue', alpha=0.7, label='Ort. Toplam Yağış (mm)')
line = ax2b.plot(rainy_days['month'], rainy_days['rainy_days'],
                 color='orange', marker='o', linewidth=2, label='Ort. Yağışlı Gün')
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(['Oca','Şub','Mar','Nis','May','Haz','Tem','Ağu','Eyl','Eki','Kas','Ara'])
ax2.set_xlabel('Ay')
ax2.set_ylabel('Toplam Yağış (mm)', color='steelblue')
ax2b.set_ylabel('Yağışlı Gün', color='orange')
ax2.set_title('Aylık Yağış Dağılımı')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}/chart7_precipitation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 7 kaydedildi.')

Grafik 7 kaydedildi.


## 10. Grafik 8 – Aktual vs Tahmin (Linear Regression)

In [16]:
# LR modeli eğit
lr = LinearRegression(featuresCol='features', labelCol=TARGET, maxIter=100)
lr_pipeline = Pipeline(stages=[imputer, indexer, assembler, lr])
lr_model = lr_pipeline.fit(train)
preds_spark = lr_model.transform(test)
preds_pd = preds_spark.select(TARGET, 'prediction').toPandas().dropna()

evaluator = RegressionEvaluator(labelCol=TARGET, predictionCol='prediction')
r2   = evaluator.setMetricName('r2').evaluate(preds_spark)
rmse = evaluator.setMetricName('rmse').evaluate(preds_spark)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Linear Regression – Tahmin Analizi (Gold: ml_features)', fontsize=12, fontweight='bold')

# Scatter: Aktual vs Tahmin
sample = preds_pd.sample(min(2000, len(preds_pd)), random_state=42)
ax1.scatter(sample[TARGET], sample['prediction'], alpha=0.3, s=8, color='steelblue')
lims = [min(sample[TARGET].min(), sample['prediction'].min()),
        max(sample[TARGET].max(), sample['prediction'].max())]
ax1.plot(lims, lims, 'r--', linewidth=1.5, label='Mükemmel Tahmin')
ax1.set_xlabel('Gerçek Sıcaklık (°C)')
ax1.set_ylabel('Tahmin Edilen Sıcaklık (°C)')
ax1.set_title(f'Aktual vs Tahmin (R²={r2:.4f}, RMSE={rmse:.3f}°C)')
ax1.legend()
ax1.grid(alpha=0.3)

# Rezidü dağılımı
residuals = preds_pd['prediction'] - preds_pd[TARGET]
ax2.hist(residuals, bins=60, color='steelblue', edgecolor='white', linewidth=0.3, alpha=0.8)
ax2.axvline(x=0,                color='red',    linestyle='-',  linewidth=1.5, label='Sıfır Rezidü')
ax2.axvline(x=residuals.mean(), color='orange', linestyle='--', linewidth=1.5, label=f'Ort: {residuals.mean():.3f}')
ax2.set_xlabel('Rezidü (Tahmin − Gerçek) °C')
ax2.set_ylabel('Frekans')
ax2.set_title('Rezidü Dağılımı')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}/chart8_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 8 kaydedildi.')

Grafik 8 kaydedildi.


## 11. Grafik 9 – Nem & Bulutluluk Korelasyonu

In [26]:
corr_cols = [
    'avg_temp_c', 'min_temp_c', 'max_temp_c',
    'temp_range_c', 'precipitation_mm',
    'season_numeric', 'temp_lag_1'
]
col_labels = {
    'avg_temp_c':       'Ort.\nSıcaklık',
    'min_temp_c':       'Min\nSıcaklık',
    'max_temp_c':       'Maks\nSıcaklık',
    'temp_range_c':     'Sıcaklık\nAralığı',
    'precipitation_mm': 'Yağış\n(mm)',
    'season_numeric':   'Mevsim',
    'temp_lag_1':       'Dünkü\nSıcaklık',
}

corr_data   = ml_pd[[c for c in corr_cols if c in ml_pd.columns]].dropna()
corr_matrix = corr_data.corr()
labels      = [col_labels.get(c, c) for c in corr_matrix.columns]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Korelasyon Katsayısı', fontsize=10)

n = len(corr_matrix)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(n):
    for j in range(n):
        val = corr_matrix.iloc[i, j]
        txt_color = 'white' if abs(val) > 0.6 else 'black'
        fw = 'bold' if i == j else 'normal'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=9, color=txt_color, fontweight=fw)

ax.set_title('Değişken Korelasyon Matrisi (Gold: ml_features)', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(f'{OUT}/chart9_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik 9 kaydedildi.')

Grafik 9 kaydedildi.


## 12. Grand Dashboard – 3×3 Özet

In [27]:
saved_charts = [
    (f'{OUT}/chart1_model_performance.png',   'Model Performansı'),
    (f'{OUT}/chart2_feature_importance.png',  'Feature Importance'),
    (f'{OUT}/chart3_yearly_trend.png',        'Yıllık Sıcaklık Trendi'),
    (f'{OUT}/chart4_monthly_seasonality.png', 'Aylık Mevsimsellik'),
    (f'{OUT}/chart5_temp_distribution.png',   'Sıcaklık Dağılımı'),
    (f'{OUT}/chart6_season_distribution.png', 'Mevsim Dağılımı'),
    (f'{OUT}/chart7_precipitation.png',       'Yağış Analizi'),
    (f'{OUT}/chart8_actual_vs_pred.png',      'Aktual vs Tahmin'),
    (f'{OUT}/chart9_correlation_matrix.png',  'Korelasyon Matrisi'),
]

fig = plt.figure(figsize=(24, 18))
fig.suptitle(
    'Daily Climate Data – Büyük Veri Analizi Dashboard\n'
    'Asadabad İstasyonu, Afganistan (1957–2010) | Gold Layer | Delta Lake',
    fontsize=16, fontweight='bold', y=0.98
)

for idx, (path, title) in enumerate(saved_charts):
    ax = fig.add_subplot(3, 3, idx + 1)
    if os.path.exists(path):
        img = plt.imread(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=10, fontweight='bold', pad=4)
    else:
        ax.text(0.5, 0.5, f'{title}\n(grafik yok)', ha='center', va='center',
                fontsize=10, transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f'{OUT}/grand_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grand Dashboard kaydedildi: grand_dashboard.png')

Grand Dashboard kaydedildi: grand_dashboard.png


## 13. Özet

In [19]:
print('=' * 60)
print('ADİM 7 – VİZUALİZASYON DASHBOARD TAMAMLANDI')
print('=' * 60)
print()
print('Kullanılan Gold Tablolar:')
print('  gold/model_results       → Grafik 1 (Model Performansı)')
print('  gold/ml_features         → Grafik 2, 5, 6, 8, 9')
print('  gold/city_monthly_summary→ Grafik 3, 4, 7')
print()
print('Kaydedilen Grafikler:')
for path, title in saved_charts:
    status = '✓' if os.path.exists(path) else '✗'
    print(f'  {status} {title:30s} → {os.path.basename(path)}')
print(f'  ✓ Grand Dashboard              → grand_dashboard.png')

spark.stop()
print()
print('SparkSession kapatıldı.')

ADİM 7 – VİZUALİZASYON DASHBOARD TAMAMLANDI

Kullanılan Gold Tablolar:
  gold/model_results       → Grafik 1 (Model Performansı)
  gold/ml_features         → Grafik 2, 5, 6, 8, 9
  gold/city_monthly_summary→ Grafik 3, 4, 7

Kaydedilen Grafikler:
  ✓ Model Performansı              → chart1_model_performance.png
  ✓ Feature Importance             → chart2_feature_importance.png
  ✓ Yıllık Sıcaklık Trendi         → chart3_yearly_trend.png
  ✓ Aylık Mevsimsellik             → chart4_monthly_seasonality.png
  ✓ Sıcaklık Dağılımı              → chart5_temp_distribution.png
  ✓ Mevsim Dağılımı                → chart6_season_distribution.png
  ✓ Yağış Analizi                  → chart7_precipitation.png
  ✓ Aktual vs Tahmin               → chart8_actual_vs_pred.png
  ✓ Korelasyon Matrisi             → chart9_correlation_matrix.png
  ✓ Grand Dashboard              → grand_dashboard.png

SparkSession kapatıldı.
